In [1]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

In [2]:
model_path = "huggingFaceOfNabil/SmolVLM2-256M-Video-Instruct-dense-caption_full"
processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    _attn_implementation="flash_attention_2"
).to("cuda")

[2025-06-17 02:58:59,806] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


[2025-06-17 02:59:00,891] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


In [30]:
#96-101
from datasets import load_dataset

test_ds = load_dataset('json', data_files="test.jsonl", split='train')
test_ds[0]['conversations'][0]['value'][8:]

'This video segment is a part of a long educational video on Database Systems: Transaction, Concurrency Control & Recovery\nBegining Clip: The lecture is being introduced.\n\nCurrent Segment Transcript: "Okay, so we will start a new chapter today, which is Transaction, Concurrency Control, and Recovery. So, transaction is an important theoretical concept in our database, and not just theoretical, it is also implemented practically, but we will look at it theoretically in our course, how any database transactions are managed."\n\nUser Query: Based on the provided context, I need you to do two things for this video segment: \n1. Provide a detailed analysis of the visual information and the educational concept being taught.\n2. Pinpoint the exact start and end times for this specific segment.'

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "video", "path": "videos/000_6oEbAmMMV1c.mp4"},
            {"type": "text", "text": "This video segment is a part of a long educational video on Database Systems: Transaction, Concurrency Control & Recovery\nBegining Clip: The lecture is being introduced.\n\nCurrent Segment Transcript: \"Okay, so we will start a new chapter today, which is Transaction, Concurrency Control, and Recovery. So, transaction is an important theoretical concept in our database, and not just theoretical, it is also implemented practically, but we will look at it theoretically in our course, how any database transactions are managed.\"\n\nUser Query: Based on the provided context, I need you to do two things for this video segment: \n1. Provide a detailed analysis of the visual information and the educational concept being taught.\n2. Pinpoint the exact start and end times for this specific segment."}
        ]
    },
]

inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=64)
generated_texts = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True,
)

print(generated_texts[0])